## 1. Cargar los datos

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path().resolve().parent
DATA_PATH = ROOT / "data_raw"

print("Directorio actual:", ROOT)
print("Ruta de datos:", DATA_PATH)

Directorio actual: C:\Users\erika\OneDrive\Escritorio\DAFT2026\Data Analitycs\SEMANA5\Projecto-2-Vanguard-ab-test
Ruta de datos: C:\Users\erika\OneDrive\Escritorio\DAFT2026\Data Analitycs\SEMANA5\Projecto-2-Vanguard-ab-test\data_raw


In [5]:
#Descarga de datos otra vez.
df_demo        = pd.read_csv(DATA_PATH / "df_final_demo.txt")
df_experiment  = pd.read_csv(DATA_PATH / "df_final_experiment_clients.txt")
df_web_pt1     = pd.read_csv(DATA_PATH / "df_final_web_data_pt_1.txt")
df_web_pt2     = pd.read_csv(DATA_PATH / "df_final_web_data_pt_2.txt")

print("Datos cargados correctamente!")

Datos cargados correctamente!


## 2. Eliminación de duplicados:

In [6]:
#Confirmamos los duplicados exactos que hay otra vez
print("Duplicados en df_demo:      ", df_demo.duplicated().sum())
print("Duplicados en df_experiment:", df_experiment.duplicated().sum())
print("Duplicados en df_web_pt1:   ", df_web_pt1.duplicated().sum())
print("Duplicados en df_web_pt2:   ", df_web_pt2.duplicated().sum())

Duplicados en df_demo:       0
Duplicados en df_experiment: 0
Duplicados en df_web_pt1:    2095
Duplicados en df_web_pt2:    8669


In [7]:
#Ahora los eliminamos.
df_web_pt1 = df_web_pt1.drop_duplicates()
df_web_pt2 = df_web_pt2.drop_duplicates()

print("Duplicados en df_web_pt1 tras limpiar:", df_web_pt1.duplicated().sum())
print("Duplicados en df_web_pt2 tras limpiar:", df_web_pt2.duplicated().sum())

Duplicados en df_web_pt1 tras limpiar: 0
Duplicados en df_web_pt2 tras limpiar: 0


## 3. Unión de df_web_pt1 y df_web_pt2

In [8]:
df_web = pd.concat([df_web_pt1, df_web_pt2], ignore_index=True)

print("Filas df_web_pt1:", len(df_web_pt1))
print("Filas df_web_pt2:", len(df_web_pt2))
print("Filas df_web total:", len(df_web))

Filas df_web_pt1: 341046
Filas df_web_pt2: 403595
Filas df_web total: 744641


## 4. Tratar nulos de df_demo

In [9]:
print("Nulos en df_demo:")
print(df_demo.isna().sum())

Nulos en df_demo:
client_id            0
clnt_tenure_yr      14
clnt_tenure_mnth    14
clnt_age            15
gendr               14
num_accts           14
bal                 14
calls_6_mnth        14
logons_6_mnth       14
dtype: int64


Muy poco datos nulos (14-15 líneas por columna). Opciones de tratamiento de nulos:
-A: Eliminar las filas con nulos (perdemos las 15 filas)(La mejor porque realmente son muy pocos datos que se "pierden")
-B: Rellenar con la media/mediana (c.numéricas) o moda en (c.categóricas)
-C:Dejarlo como estña (No recomendable porque darían problemas en los analisis)

In [10]:
#A: Eliminación de nulos.
df_demo = df_demo.dropna()

print("Filas antes de limpiar: 70609")
print("Filas después de limpiar:", len(df_demo))
print("Nulos restantes:")
print(df_demo.isna().sum())

Filas antes de limpiar: 70609
Filas después de limpiar: 70594
Nulos restantes:
client_id           0
clnt_tenure_yr      0
clnt_tenure_mnth    0
clnt_age            0
gendr               0
num_accts           0
bal                 0
calls_6_mnth        0
logons_6_mnth       0
dtype: int64


## 5. Tratar nulos en df_experiment

In [11]:
print("Nulos en df_experiment:")
print(df_experiment.isna().sum())

Nulos en df_experiment:
client_id        0
Variation    20109
dtype: int64


En df_experiment, tenemos 20109 datos nulos en la columna "Variation", que se entiende que hubo 20109 clientes que no participaron en el experimento. Así que hay dos opciones:
-A: Eliminarlos (Pero aquí si que perderiamos muchos datos, no es lo recomendable)
-B: Filtrar y quedarnos solo con los clientes que sí participaron en el experimento.

In [12]:
#Opción B: Filtrar columna "Variation" en df_experiment
df_experiment = df_experiment.dropna(subset=["Variation"])

print("Clientes en el experimento:", len(df_experiment))
print("Nulos restantes:")
print(df_experiment.isna().sum())
print()
print("Distribución Test vs Control:")
print(df_experiment["Variation"].value_counts())

Clientes en el experimento: 50500
Nulos restantes:
client_id    0
Variation    0
dtype: int64

Distribución Test vs Control:
Variation
Test       26968
Control    23532
Name: count, dtype: int64


Ahora tenemos 50.500 clientes que participaron en el experimento, de los cuales 26.968 están en Test y 23.532 en Control. Esto quiere decir que ahora los datos están bastante equilibrados.

## 6. Convertir data_time a datetime
Necesitamos convertir la columna date_time de df_web que está en texto a un formato fecha y hora para que Python lo entienda, para poder calucular el tiempo que los usuarios han pasado en cada paso

In [13]:
df_web["date_time"] = pd.to_datetime(df_web["date_time"])

print("Tipo de dato antes:", "object (string)")
print("Tipo de dato después:", df_web["date_time"].dtype)
print()
print("Ejemplo de fechas:")
print(df_web["date_time"].head())

Tipo de dato antes: object (string)
Tipo de dato después: datetime64[ns]

Ejemplo de fechas:
0   2017-04-17 15:27:07
1   2017-04-17 15:26:51
2   2017-04-17 15:19:22
3   2017-04-17 15:19:13
4   2017-04-17 15:18:04
Name: date_time, dtype: datetime64[ns]


## 7. Guardar los datasets limpios

In [14]:
df_demo.to_csv(DATA_PATH / "df_demo_clean.csv", index=False)
df_experiment.to_csv(DATA_PATH / "df_experiment_clean.csv", index=False)
df_web.to_csv(DATA_PATH / "df_web_clean.csv", index=False)

print("Datasets guardados correctamente!")
print("df_demo_clean:      ", len(df_demo), "filas")
print("df_experiment_clean:", len(df_experiment), "filas")
print("df_web_clean:       ", len(df_web), "filas")

Datasets guardados correctamente!
df_demo_clean:       70594 filas
df_experiment_clean: 50500 filas
df_web_clean:        744641 filas
